## **Road Pulse:** Feature Engineering

### Library Imports

In [24]:
import joblib 
import numpy as np
import pandas as pd

### **Loading data**

In [25]:
def load_data():
    loaded_data = []

    for i in range(7):
        data = pd.read_excel(f'../data/raw/nez-opendata-202{i}.xlsx')
        data.columns = ['accident_id', 'department', 'municipality', 'date_time', 'longitude', 'latitude', 'accident_type', 'involved_vehicles_num', 'description']

        data['date_time'] = pd.to_datetime(
            data['date_time'],
            format="%d.%m.%Y,%H:%M", 
            errors='coerce'
        )

        loaded_data.append(data)

    return pd.concat(loaded_data)

In [26]:
data = load_data()

In [27]:
data.to_excel('../data/raw/nez-opendata-all.xlsx', index=False)

In [28]:
data.head()

,accident_id,department,municipality,date_time,longitude,latitude,accident_type,involved_vehicles_num,description
0,1278576,BEOGRAD,BARAJEVO,2020-01-07 10:10:00,20.301589,44.568563,Sa mat.stetom,SN SA JEDNIM VOZILOM,Nezgoda sa jednim vozilom – silazak sa kolovoz...
1,1278948,BEOGRAD,BARAJEVO,2020-01-09 12:50:00,20.413280,44.579780,Sa mat.stetom,SN SA NAJMANjE DVA VOZILA – BEZ SKRETANjA,Najmanje dva vozila koja se kreću u istom smer...
2,1278074,BEOGRAD,BARAJEVO,2020-01-10 10:05:00,20.312560,44.575470,Sa mat.stetom,SN SA NAJMANjE DVA VOZILA – SKRETANjE ILI PREL...,Najmanje dva vozila koja se kreću istim putem ...
3,1278582,BEOGRAD,BARAJEVO,2020-01-10 15:15:00,20.412190,44.637770,Sa mat.stetom,SN SA NAJMANjE DVA VOZILA – SKRETANjE ILI PREL...,Najmanje dva vozila koja se kreću različitim p...
4,1278638,BEOGRAD,BARAJEVO,2020-01-16 16:45:00,20.381686,44.633030,Sa povredjenim,SN SA NAJMANjE DVA VOZILA – BEZ SKRETANjA,Najmanje dva vozila koja se kreću u istom smer...


### **Feature engineering**

#### **Feature:** `municipality`

In [5]:
import category_encoders as ce

In [ ]:
municipality_encoder = ce.TargetEncoder(cols=['municipality'])
data['municipality_encoded'] = municipality_encoder.fit_transform(data['municipality'], data['accident_type'])

In [8]:
data.head(3)

,accident_id,department,municipality,date_time,longitude,latitude,accident_type,involved_vehicles_num,description,municipality_encoded
0,1278576,BEOGRAD,BARAJEVO,2020-01-07 10:10:00,20.301589,44.568563,Sa mat.stetom,SN SA JEDNIM VOZILOM,Nezgoda sa jednim vozilom – silazak sa kolovoz...,0.698704
1,1278948,BEOGRAD,BARAJEVO,2020-01-09 12:50:00,20.413280,44.579780,Sa mat.stetom,SN SA NAJMANjE DVA VOZILA – BEZ SKRETANjA,Najmanje dva vozila koja se kreću u istom smer...,0.698704
2,1278074,BEOGRAD,BARAJEVO,2020-01-10 10:05:00,20.312560,44.575470,Sa mat.stetom,SN SA NAJMANjE DVA VOZILA – SKRETANjE ILI PREL...,Najmanje dva vozila koja se kreću istim putem ...,0.698704


#### **Feature:** `date_time`

New features:
- `year`
- `month`
- `day_of_week`
- `hour`
- `day_type` (Weekday or Weekend)
- `is_rush`
- `is_night`

In [29]:
data['year'] = data['date_time'].dt.year
data['month'] = data['date_time'].dt.month
data['day_of_week'] = data['date_time'].dt.day_of_week
data['hour'] = data['date_time'].dt.hour

data = data.drop(columns='date_time')

In [30]:
data['day_type'] = np.where((data['day_of_week'] == 5) | (data['day_of_week'] == 6), 0, 1)

In [31]:
data['is_rush'] = data['hour'].isin([7, 8, 16, 17, 18]).astype(int)

In [32]:
data['is_night'] = (data['hour'].between(22, 23) | data['hour'].between(0, 6)).astype(int)

In [33]:
def get_season(month):
    if month in [12, 1, 2]:
        return 0
    elif month in [3, 4, 5]:
        return 1
    elif month in [6, 7, 8]:
        return 2
    else:
        return 3

In [34]:
data['season'] = data['month'].apply(get_season)

In [35]:
data.head(3)

,accident_id,department,municipality,longitude,latitude,accident_type,involved_vehicles_num,description,year,month,day_of_week,hour,day_type,is_rush,is_night,season
0,1278576,BEOGRAD,BARAJEVO,20.301589,44.568563,Sa mat.stetom,SN SA JEDNIM VOZILOM,Nezgoda sa jednim vozilom – silazak sa kolovoz...,2020,1,1,10,1,0,0,0
1,1278948,BEOGRAD,BARAJEVO,20.413280,44.579780,Sa mat.stetom,SN SA NAJMANjE DVA VOZILA – BEZ SKRETANjA,Najmanje dva vozila koja se kreću u istom smer...,2020,1,3,12,1,0,0,0
2,1278074,BEOGRAD,BARAJEVO,20.312560,44.575470,Sa mat.stetom,SN SA NAJMANjE DVA VOZILA – SKRETANjE ILI PREL...,Najmanje dva vozila koja se kreću istim putem ...,2020,1,4,10,1,0,0,0


#### **Feature:** `accident_type`

In [36]:
mapping = {
    'Sa mat.stetom': 'material',
    'Sa povredjenim': 'injured',
    'Sa poginulim': 'dead',
}

data['accident_type'] = data['accident_type'].map(mapping)

# 0 = material
# 1 = injured or dead
data['accident_type'] = (data['accident_type'] != 'material').astype(int)

data.groupby('accident_type').size()

accident_type
0    122545
1     82807
dtype: int64

In [37]:
data['accident_type'].value_counts(normalize=True)

accident_type
0    0.596756
1    0.403244
Name: proportion, dtype: float64

In [38]:
data.head(3)

,accident_id,department,municipality,longitude,latitude,accident_type,involved_vehicles_num,description,year,month,day_of_week,hour,day_type,is_rush,is_night,season
0,1278576,BEOGRAD,BARAJEVO,20.301589,44.568563,0,SN SA JEDNIM VOZILOM,Nezgoda sa jednim vozilom – silazak sa kolovoz...,2020,1,1,10,1,0,0,0
1,1278948,BEOGRAD,BARAJEVO,20.413280,44.579780,0,SN SA NAJMANjE DVA VOZILA – BEZ SKRETANjA,Najmanje dva vozila koja se kreću u istom smer...,2020,1,3,12,1,0,0,0
2,1278074,BEOGRAD,BARAJEVO,20.312560,44.575470,0,SN SA NAJMANjE DVA VOZILA – SKRETANjE ILI PREL...,Najmanje dva vozila koja se kreću istim putem ...,2020,1,4,10,1,0,0,0


#### **Feature:** `involved_vehicle_number`

In [39]:
data['involved_vehicles_num'].unique()

<StringArray>
[                              'SN SA JEDNIM VOZILOM',
          'SN SA NAJMANjE DVA VOZILA – BEZ SKRETANjA',
 'SN SA NAJMANjE DVA VOZILA – SKRETANjE ILI PRELAZAK',
                          'SN SA PARKIRANIM VOZILIMA',
                                     'SN SA PEŠACIMA']
Length: 5, dtype: str

In [40]:
def preprocess_involved_vehicles_num(df, column='involved_vehicles_num'):

    mapping = {
        r'SN SA JEDNIM VOZILOM': 'single_vehicle',
        r'SN SA NAJMANjE DVA VOZILA – BEZ SKRETANjA': 'two_vehicles_no_turn',
        r'SN SA NAJMANjE DVA VOZILA – SKRETANjE ILI PRELAZAK': 'two_vehicles_turn_or_cross',
        r'SN SA PARKIRANIM VOZILIMA': 'parked_vehicles',
        r'SN SA PEŠACIMA': 'pedestrians'
    }

    df[column] = df[column].astype(str).str.strip()

    df[column] = df[column].replace(mapping, regex=True)

    return df    

Binary Flag Decomposition:

In [41]:
def extract_involved_vehicles_column(df, column='involved_vehicles_num'):
    columns = pd.DataFrame(index=df.index)

    values = df[column].fillna('').str.lower()

    columns['single_vehicle'] = values.str.contains('jednim vozilom').astype(int)

    columns['multiple_vehicles'] = values.str.contains('najmanje dva vozila').astype(int)
    
    columns['parked_vehicles'] = values.str.contains('parkiranim vozilima').astype(int)

    columns['pedestrian'] = values.str.contains('pešacima').astype(int)

    columns['turning_crossing'] = values.str.contains('spretanje|prelazak').astype(int)

    columns['no_turning'] = values.str.contains('bez skretanja').astype(int)

    return columns

> Instead of 5 categories we create multiple binary categories where each represents one characteristic of the accident.

One-Hot Encoding:

In [42]:
data = preprocess_involved_vehicles_num(df=data)

In [43]:
dummies = pd.get_dummies(data['involved_vehicles_num'], prefix='acc', dtype=int)
data = pd.concat([data, dummies], axis=1)

In [44]:
data.drop(columns='involved_vehicles_num', inplace=True)
data.head(3)

,accident_id,department,municipality,longitude,latitude,accident_type,description,year,month,day_of_week,hour,day_type,is_rush,is_night,season,acc_parked_vehicles,acc_pedestrians,acc_single_vehicle,acc_two_vehicles_no_turn,acc_two_vehicles_turn_or_cross
0,1278576,BEOGRAD,BARAJEVO,20.301589,44.568563,0,Nezgoda sa jednim vozilom – silazak sa kolovoz...,2020,1,1,10,1,0,0,0,0,0,1,0,0
1,1278948,BEOGRAD,BARAJEVO,20.413280,44.579780,0,Najmanje dva vozila koja se kreću u istom smer...,2020,1,3,12,1,0,0,0,0,0,0,1,0
2,1278074,BEOGRAD,BARAJEVO,20.312560,44.575470,0,Najmanje dva vozila koja se kreću istim putem ...,2020,1,4,10,1,0,0,0,0,0,0,0,1


#### **Feature:** `description`

In [45]:
data['description'].unique()

<StringArray>
[                                                                                  'Nezgoda sa jednim vozilom – silazak sa kolovoza u krivini',
                                                                  'Najmanje dva vozila koja se kreću u istom smeru – uključivanje u saobraćaj',
                                 'Najmanje dva vozila koja se kreću istim putem u suprotnim smerovima uz skretanje ulevo ispred drugog vozila',
                                  'Najmanje dva vozila koja se kreću različitim putevima uz skretanje udesno ispred vozila koje nailazi sleva',
                                                                                'Najmanje dva vozila koja se kreću u istom smeru – sustizanje',
                                                                            'Nezgoda sa jednim vozilom – silazak udesno sa kolovoza na pravcu',
                                                  'Ostale nezgode sa najmanje dva vozila koja se kreću istim putem u istom

Label Encoding:

In [ ]:
description_encoder = ce.TargetEncoder(cols=['description'])
data['description_encoded'] = description_encoder.fit_transform(data['description'], data['accident_type'])

In [27]:
data.head(3)

,accident_id,department,municipality,longitude,latitude,accident_type,description,municipality_encoded,year,month,...,day_type,is_rush,is_night,season,acc_parked_vehicles,acc_pedestrians,acc_single_vehicle,acc_two_vehicles_no_turn,acc_two_vehicles_turn_or_cross,description_encoded
0,1278576,BEOGRAD,BARAJEVO,20.301589,44.568563,0,Nezgoda sa jednim vozilom – silazak sa kolovoz...,0.698704,2020,1,...,1,0,0,0,0,0,1,0,0,0.541965
1,1278948,BEOGRAD,BARAJEVO,20.413280,44.579780,0,Najmanje dva vozila koja se kreću u istom smer...,0.698704,2020,1,...,1,0,0,0,0,0,0,1,0,0.276762
2,1278074,BEOGRAD,BARAJEVO,20.312560,44.575470,0,Najmanje dva vozila koja se kreću istim putem ...,0.698704,2020,1,...,1,0,0,0,0,0,0,0,1,0.610821


In [46]:
def preprocessed_info(df):
    print(f'Number of Features: {len(df.columns)}')
    print('Features:')
    print(*df.columns, sep='\n')

In [47]:
preprocessed_info(df=data)

Number of Features: 20
Features:
accident_id
department
municipality
longitude
latitude
accident_type
description
year
month
day_of_week
hour
day_type
is_rush
is_night
season
acc_parked_vehicles
acc_pedestrians
acc_single_vehicle
acc_two_vehicles_no_turn
acc_two_vehicles_turn_or_cross


In [48]:
data.drop(columns=['accident_id', 'department', 'year', 'description'], inplace=True)

In [49]:
preprocessed_info(df=data)

Number of Features: 16
Features:
municipality
longitude
latitude
accident_type
month
day_of_week
hour
day_type
is_rush
is_night
season
acc_parked_vehicles
acc_pedestrians
acc_single_vehicle
acc_two_vehicles_no_turn
acc_two_vehicles_turn_or_cross


### **Saving preprocessed data**

In [50]:
data.to_excel('../data/processed/accident_processed_srb.xlsx', index=False)

### **Saving label encoders**

In [33]:
joblib.dump(municipality_encoder, '../models/encoders/le_municipality.pkl')

['../models/encoders/le_municipality.pkl']

In [34]:
joblib.dump(description_encoder, '../models/encoders/le_description.pkl')

['../models/encoders/le_description.pkl']